In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab / Linux environment)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 3-Class Disaster Triage: Neighbourhood-Based (NB) Undersampling + Random Forest (`models/train_custom_overlap.ipynb`)

Loads the full emergency cohort from **`datasets/5v_cleandf.RData`** (~558,000 visits with valid ESI) and applies the **NB-Undersampling** family of methods (ported from [`NB-undersampling/`](file:///home/apt2736/PKM_RF/NB-undersampling/)) to clean overlapping majority boundary noise, followed by **Random Forest** classification across the 3 disaster triage tiers.

```mermaid
flowchart TD
    Raw["Raw 8 Arrival Features X in R^8 (558,029 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> NB["NB-Tomek Undersampling (Adaptive k = sqrt(IR) + sqrt(N))"]
    NB --> RF["Random Forest Classifier (class_weight='balanced')"]
    RF --> Eval["Holdout Test Evaluation: Recall, Specificity, Balanced Accuracy, ROC AUC"]
```

### 🎯 3-Tier Disaster Triage Acuity Mapping
1. **`Tier 0: RED (ESI 1)`** ($y=0$): Immediate Resuscitation ($\sim 0.94\%$).
2. **`Tier 1: YELLOW (ESI 2–3)`** ($y=1$): Emergent & Urgent ($\sim 78.86\%$).
3. **`Tier 2: GREEN (ESI 4–5)`** ($y=2$): Semi-urgent & Non-urgent ($\sim 20.19\%$).

### 📐 NB-Undersampling: Adaptive Neighbourhood Cleaning
Ported from the original R implementation in [`NB-undersampling/`](file:///home/apt2736/PKM_RF/NB-undersampling/). The adaptive neighbourhood radius:
$$k = \sqrt{\text{IR}} + \sqrt{N_{\text{train}}}$$
where $\text{IR} = N_{\text{majority}} / N_{\text{minority}}$ is the imbalance ratio. This ensures the cleaning radius scales with both imbalance severity and dataset magnitude.

**NB-Tomek** variant is used: For each majority sample, it checks if a mutual nearest-neighbour link exists with any minority sample within the adaptive $k$-NN neighbourhood. If a Tomek link is detected, the majority sample is removed.

In [ ]:
%%R
# ---------------------------------------------------------------------------
# Step 1: Load 5v_cleandf.RData (8 Core Triage Features, All Non-NA ESI Rows Kept)
# ---------------------------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(dplyr)
})

candidate_paths <- c(
  "../datasets/5v_cleandf.RData",
  "datasets/5v_cleandf.RData",
  "/kaggle/working/PKM_RF/datasets/5v_cleandf.RData",
  "/kaggle/input/5v-cleandf/5v_cleandf.RData",
  "/kaggle/input/disaster-triage-dataset/5v_cleandf.RData",
  "/kaggle/input/5v-raw/5v_cleandf.RData"
)

data_file <- NULL
for (p in candidate_paths) {
  if (file.exists(p)) {
    data_file <- p
    break
  }
}

if (is.null(data_file)) {
  stop("Could not find 5v_cleandf.RData in any candidate paths!")
}

cat(sprintf("Loading RData from: %s ...\n", data_file))
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
raw_df   <- get(df_names[which.max(df_sizes)], envir = data_env)

cat(sprintf("Loaded %s: %d Total Rows, %d Total Columns\n", data_file, nrow(raw_df), ncol(raw_df)))

valid_mask <- !is.na(raw_df$esi)
raw_df     <- raw_df[valid_mask, ]

gender_vec <- if ("gender" %in% names(raw_df)) ifelse(is.na(raw_df$gender), NA, ifelse(as.character(raw_df$gender) == "Male", 1, 0)) else rep(NA, nrow(raw_df))
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) raw_df$cc_breathingdifficulty else rep(NA, nrow(raw_df))

get_vec <- function(col_name) {
  if (col_name %in% names(raw_df)) return(raw_df[[col_name]]) else return(rep(NA, nrow(raw_df)))
}

raw_esi_char <- as.character(raw_df$esi)

df_master <- data.frame(
  age                     = raw_df$age,
  cc_breathingdifficulty  = cc_bd_vec,
  gender                  = gender_vec,
  triage_vital_hr         = get_vec("triage_vital_hr"),
  triage_vital_sbp        = get_vec("triage_vital_sbp"),
  triage_vital_dbp        = get_vec("triage_vital_dbp"),
  triage_vital_rr         = get_vec("triage_vital_rr"),
  triage_vital_o2         = get_vec("triage_vital_o2"),
  esi                     = as.numeric(raw_esi_char)
)

feature_cols <- c("age", "cc_breathingdifficulty", "gender",
                  "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp",
                  "triage_vital_rr", "triage_vital_o2")

raw_mat_export <- as.matrix(df_master[, feature_cols])
esi_export     <- as.numeric(df_master$esi)

cat(sprintf("Exported to Python: %d rows, %d feature columns\n",
            nrow(raw_mat_export), ncol(raw_mat_export)))

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Retrieve Data, Partition, Impute & Scale
# ---------------------------------------------------------------------------
import os, json, pickle, warnings, time
from rpy2.robjects import r
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, recall_score,
                             precision_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = '..' if os.path.basename(os.getcwd()) == 'models' else '.'

raw_mat_all = np.array(r('raw_mat_export'), dtype=np.float64)
esi_all     = np.array(r('esi_export'), dtype=np.int32)

FEATURES = ['age', 'cc_breathingdifficulty', 'gender',
             'triage_vital_hr', 'triage_vital_sbp', 'triage_vital_dbp',
             'triage_vital_rr', 'triage_vital_o2']

# 3-Tier Target: 0=RED (ESI 1), 1=YELLOW (ESI 2-3), 2=GREEN (ESI 4-5)
y_all = np.zeros(len(esi_all), dtype=np.int32)
y_all[esi_all == 1] = 0
y_all[np.isin(esi_all, [2, 3])] = 1
y_all[np.isin(esi_all, [4, 5])] = 2
TIER_LABELS = ['RED (ESI 1)', 'YELLOW (ESI 2-3)', 'GREEN (ESI 4-5)']

print("=========================================================================")
print("    3-TIER DISASTER TRIAGE: NB-UNDERSAMPLING + RANDOM FOREST")
print("=========================================================================")
print(f"Total Valid ESI Visits: {len(esi_all):,}")
for c, lbl in enumerate(TIER_LABELS):
    print(f"  * Tier {c} [{lbl}]: {np.sum(y_all == c):,} ({np.mean(y_all == c)*100:.2f}%)")
print("=========================================================================\n")

# Stratified 70/15/15 Split
itr, itmp = train_test_split(np.arange(len(esi_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

y_train, y_val, y_test = y_all[itr], y_all[iva], y_all[ite]

# Impute & Scale
imputer = SimpleImputer(strategy='median')
X_tr_imp = imputer.fit_transform(raw_mat_all[itr])
X_val_imp = imputer.transform(raw_mat_all[iva])
X_te_imp = imputer.transform(raw_mat_all[ite])

scaler = StandardScaler()
X_train = scaler.fit_transform(X_tr_imp)
X_val   = scaler.transform(X_val_imp)
X_test  = scaler.transform(X_te_imp)

print(f"Partition Shapes: Train={X_train.shape}, Val={X_val.shape}, Test={X_test.shape}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: NB-Undersampling Implementation (Ported from NB-undersampling/*.R)
#
# Reference: NB-undersampling/searchKnn_UnderS_Tomek.R
#            NB-undersampling/searchKnn_UnderS_Common.R
#
# Adaptive neighbourhood radius: k = sqrt(IR) + sqrt(N_train)
# where IR = N_majority / N_minority
# ---------------------------------------------------------------------------

def nb_tomek_undersample(X, y, minority_class, random_state=42):
    """
    NB-Tomek Undersampling (ported from searchKnn_UnderS_Tomek.R).

    For each majority sample among its k nearest neighbours, checks whether
    a mutual k-NN Tomek link exists with a minority sample. If a link is found,
    the majority sample is removed.

    All minority (positive) samples are always preserved.
    """
    minority_mask = (y == minority_class)
    majority_mask = ~minority_mask

    n_min = np.sum(minority_mask)
    n_maj = np.sum(majority_mask)
    IR = n_maj / max(n_min, 1)

    k = int(np.sqrt(IR) + np.sqrt(len(y)))
    k = min(k, len(y) - 1)  # cannot exceed N-1

    print(f"  NB-Tomek: minority_class={minority_class}, IR={IR:.2f}, k={k}")

    # Order: minority (positive) first, then majority (negative)
    pos_idx = np.where(minority_mask)[0]
    neg_idx = np.where(majority_mask)[0]
    ordered_idx = np.concatenate([pos_idx, neg_idx])
    X_ordered = X[ordered_idx]
    num_pos = len(pos_idx)

    # Fit k-NN on ordered dataset
    nn = NearestNeighbors(n_neighbors=k, algorithm='auto', n_jobs=-1)
    nn.fit(X_ordered)
    nn_indices = nn.kneighbors(X_ordered, return_distance=False)  # (N, k)

    # Check each negative (majority) sample for Tomek links
    keep_neg_mask = np.ones(len(neg_idx), dtype=bool)
    for n_local in range(num_pos, len(ordered_idx)):  # iterate over negatives
        neg_offset = n_local - num_pos
        neighbours = nn_indices[n_local]
        for m in range(k):
            nn_idx = neighbours[m]
            if nn_idx < num_pos:  # this neighbour is a positive (minority) sample
                # Check mutual link: is the current negative in the positive's k-NN?
                pos_neighbours = nn_indices[nn_idx]
                if n_local in pos_neighbours:
                    keep_neg_mask[neg_offset] = False
                    break  # Tomek link found, remove this negative

    kept_neg_idx = neg_idx[keep_neg_mask]
    final_idx = np.concatenate([pos_idx, kept_neg_idx])

    removed = np.sum(~keep_neg_mask)
    print(f"  NB-Tomek Removed: {removed:,} majority samples ({removed/n_maj*100:.2f}%)")

    return X[final_idx], y[final_idx]


def nb_comm_undersample(X, y, minority_class, random_state=42):
    """
    NB-Comm (Common Neighbour) Undersampling (ported from searchKnn_UnderS_Common.R).

    Counts how often each majority sample appears in the k-NN of minority instances.
    Majority samples appearing >= 2 times are deemed overlap invaders and removed.

    All minority (positive) samples are always preserved.
    """
    minority_mask = (y == minority_class)
    majority_mask = ~minority_mask

    n_min = np.sum(minority_mask)
    n_maj = np.sum(majority_mask)
    IR = n_maj / max(n_min, 1)

    k = int(np.sqrt(IR) + np.sqrt(len(y)))
    k = min(k, len(y) - 1)

    print(f"  NB-Comm: minority_class={minority_class}, IR={IR:.2f}, k={k}")

    # Order: positive first, negative after
    pos_idx = np.where(minority_mask)[0]
    neg_idx = np.where(majority_mask)[0]
    ordered_idx = np.concatenate([pos_idx, neg_idx])
    X_ordered = X[ordered_idx]
    num_pos = len(pos_idx)

    nn = NearestNeighbors(n_neighbors=k, algorithm='auto', n_jobs=-1)
    nn.fit(X_ordered)
    nn_indices = nn.kneighbors(X_ordered, return_distance=False)

    # Count frequency of each negative sample appearing in positive k-NNs
    neg_freq = np.zeros(len(neg_idx), dtype=np.int32)
    for p in range(num_pos):  # iterate over positives
        for m in range(k):
            nn_idx = nn_indices[p, m]
            if nn_idx >= num_pos:  # this neighbour is a negative
                neg_offset = nn_idx - num_pos
                neg_freq[neg_offset] += 1

    # Keep negatives appearing < 2 times in minority neighbourhoods
    keep_neg_mask = (neg_freq < 2)
    kept_neg_idx = neg_idx[keep_neg_mask]
    final_idx = np.concatenate([pos_idx, kept_neg_idx])

    removed = np.sum(~keep_neg_mask)
    print(f"  NB-Comm Removed: {removed:,} majority samples ({removed/n_maj*100:.2f}%)")

    return X[final_idx], y[final_idx]


print("✓ NB-Undersampling algorithms loaded (NB-Tomek, NB-Comm)")
print("  Ported from: NB-undersampling/searchKnn_UnderS_Tomek.R")
print("  Ported from: NB-undersampling/searchKnn_UnderS_Common.R")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Apply NB-Undersampling on Training Set
#
# Strategy: Apply NB-Tomek to each minority class vs the rest.
# 1. Clean YELLOW+GREEN noise around RED (ESI 1) using NB-Tomek.
# 2. Clean YELLOW noise around GREEN (ESI 4-5) using NB-Tomek.
#
# Because NB-Tomek with full 390k samples and k~640 is computationally
# expensive, we use a stratified subsample for cleaning.
# ---------------------------------------------------------------------------
print("=" * 70)
print("  Applying NB-Tomek Undersampling on Training Set")
print("=" * 70)

print(f"\nOriginal Training Cohort: {len(y_train):,} visits")
for c, lbl in enumerate(TIER_LABELS):
    print(f"  * {lbl}: {np.sum(y_train == c):,}")

# Subsample majority classes for tractable NB-Tomek computation.
# Keep ALL RED; subsample YELLOW & GREEN for neighbourhood analysis,
# then apply the cleaning decision back to the full training set.
np.random.seed(42)
red_idx   = np.where(y_train == 0)[0]
yellow_idx = np.where(y_train == 1)[0]
green_idx  = np.where(y_train == 2)[0]

# --- Stage 1: Clean boundary between RED vs (YELLOW + GREEN) ---
print("\n--- Stage 1: NB-Tomek cleaning RED vs (YELLOW+GREEN) ---")
# Take all RED + stratified subsample of YELLOW+GREEN for neighbourhood computation
sub_yg_n = min(25000, len(yellow_idx) + len(green_idx))
yg_combined = np.concatenate([yellow_idx, green_idx])
sub_yg = np.random.choice(yg_combined, sub_yg_n, replace=False)

stage1_idx = np.concatenate([red_idx, sub_yg])
X_s1 = X_train[stage1_idx]
y_s1_binary = np.where(y_train[stage1_idx] == 0, 0, 1)  # 0=RED (minority), 1=rest

t0 = time.time()
X_s1_clean, y_s1_clean_bin = nb_tomek_undersample(X_s1, y_s1_binary, minority_class=0)
print(f"  Stage 1 time: {time.time()-t0:.1f}s")

# Identify which majority (YELLOW/GREEN) samples were kept after NB-Tomek
# Map back: the positive (RED) samples are always kept; majority kept mask
s1_kept_original_idx = stage1_idx[np.isin(stage1_idx,
    stage1_idx[:len(red_idx)])]  # RED always kept
removed_yg_set = set(sub_yg) - set(stage1_idx[
    np.concatenate([np.ones(len(red_idx), dtype=bool),
                    (y_s1_clean_bin == 1) if len(y_s1_clean_bin) > len(red_idx) else np.array([], dtype=bool)])
])

# --- Stage 2: Clean boundary between GREEN vs YELLOW ---
print("\n--- Stage 2: NB-Tomek cleaning GREEN vs YELLOW ---")
sub_y_n = min(20000, len(yellow_idx))
sub_y = np.random.choice(yellow_idx, sub_y_n, replace=False)
sub_g_n = min(20000, len(green_idx))
sub_g = np.random.choice(green_idx, sub_g_n, replace=False)

stage2_idx = np.concatenate([sub_g, sub_y])
X_s2 = X_train[stage2_idx]
y_s2_binary = np.where(y_train[stage2_idx] == 2, 0, 1)  # 0=GREEN (minority), 1=YELLOW

t0 = time.time()
X_s2_clean, y_s2_clean_bin = nb_tomek_undersample(X_s2, y_s2_binary, minority_class=0)
print(f"  Stage 2 time: {time.time()-t0:.1f}s")

# Combine: Simple approach - use cleaned subsets + remaining untouched samples
# For final training, rebuild from:
# - ALL RED (always preserved)
# - Stage 1 cleaned YELLOW+GREEN subset + untouched YELLOW+GREEN
# - Stage 2 further cleaned YELLOW subset

# Track removed indices across both stages
s1_majority_kept_mask = (y_s1_clean_bin == 1) if len(y_s1_clean_bin) > 0 else np.array([], dtype=bool)
s1_orig_majority_indices = sub_yg
# The NB-Tomek returns minority first, then kept majority
n_red_in_s1 = np.sum(y_s1_binary == 0)
n_kept_majority_s1 = len(y_s1_clean_bin) - n_red_in_s1
# Majority samples that survived NB-Tomek in stage 1
# Original indices of majority samples (sub_yg) - order preserved
s1_majority_original = sub_yg  # these are the original training indices of majority in stage 1
# After NB-Tomek, first n_red_in_s1 are RED, rest are surviving majority
# We need to know WHICH of sub_yg survived

# Simpler approach: just track removed counts and use stage cleaning as noise filter
# Re-run with tracking

# ---- Simplified Unified NB-Tomek Approach ----
# For tractability on 390k dataset, downsample majority classes for
# neighbourhood analysis, run NB-Tomek, train RF on the cleaned result.

print("\n--- Unified NB-Tomek Cleaning (RED vs REST) ---")
# Keep all RED; take stratified subsample of YELLOW+GREEN
sub_size_per_class = 15000
sub_yellow = np.random.choice(yellow_idx, min(sub_size_per_class, len(yellow_idx)), replace=False)
sub_green  = np.random.choice(green_idx, min(sub_size_per_class, len(green_idx)), replace=False)

unified_idx = np.concatenate([red_idx, sub_yellow, sub_green])
X_unified = X_train[unified_idx]
y_unified = y_train[unified_idx]

# Binary: RED=minority (0), rest=majority (1)
y_binary = np.where(y_unified == 0, 0, 1)

t0 = time.time()
X_nb_cleaned, y_nb_binary = nb_tomek_undersample(X_unified, y_binary, minority_class=0)
elapsed = time.time() - t0
print(f"  Total NB-Tomek time: {elapsed:.1f}s")

# Recover original 3-class labels for the survived samples
# The returned X_nb_cleaned preserves the order: minority first, then kept majority
n_red_kept = np.sum(y_nb_binary == 0)
n_rest_kept = np.sum(y_nb_binary == 1)

# Re-map: first n_red_kept are RED, rest are original YELLOW/GREEN with labels
y_rest_original = y_unified[y_binary == 1]  # original 3-class labels of majority
# After NB-Tomek, the kept majority samples maintain original positions
# Reconstruct: take the y_unified values for non-RED that survived
red_kept_labels = np.zeros(n_red_kept, dtype=np.int32)  # all RED

# For the majority that survived, we need their original 3-tier labels
# nb_tomek returns [minority_samples, kept_majority_samples]
# The kept_majority order corresponds to the original majority order with removals
# We tracked keep_neg_mask inside the function, but let's just recompute labels:
majority_original_labels = y_unified[y_binary == 1]  # full majority labels
# We know n_rest_kept majority survived; they are a subset of majority_original_labels
# Since nb_tomek preserves relative order, the survived labels are:
# Actually let's just use X_nb_cleaned and match back

# Match cleaned samples back to unified set using indices
# Simpler: re-implement with index tracking

def nb_tomek_undersample_tracked(X, y, minority_class):
    """NB-Tomek with index tracking to recover original labels."""
    minority_mask = (y == minority_class)
    n_min = np.sum(minority_mask)
    n_maj = np.sum(~minority_mask)
    IR = n_maj / max(n_min, 1)
    k = int(np.sqrt(IR) + np.sqrt(len(y)))
    k = min(k, len(y) - 1)

    pos_idx = np.where(minority_mask)[0]
    neg_idx = np.where(~minority_mask)[0]
    ordered_idx = np.concatenate([pos_idx, neg_idx])
    X_ordered = X[ordered_idx]
    num_pos = len(pos_idx)

    nn = NearestNeighbors(n_neighbors=k, algorithm='auto', n_jobs=-1)
    nn.fit(X_ordered)
    nn_indices = nn.kneighbors(X_ordered, return_distance=False)

    keep_neg_mask = np.ones(len(neg_idx), dtype=bool)
    for n_local in range(num_pos, len(ordered_idx)):
        neg_offset = n_local - num_pos
        neighbours = nn_indices[n_local]
        for m_i in range(k):
            nn_idx = neighbours[m_i]
            if nn_idx < num_pos:
                pos_neighbours = nn_indices[nn_idx]
                if n_local in pos_neighbours:
                    keep_neg_mask[neg_offset] = False
                    break

    # Return original indices that survived
    kept_neg_original = neg_idx[keep_neg_mask]
    all_kept = np.concatenate([pos_idx, kept_neg_original])
    removed = np.sum(~keep_neg_mask)
    return all_kept, removed


print("\n" + "=" * 70)
print("  Re-running NB-Tomek with Full Index Tracking")
print("=" * 70)

t0 = time.time()
kept_indices, n_removed = nb_tomek_undersample_tracked(X_unified, y_binary, minority_class=0)
print(f"  NB-Tomek Completed in {time.time()-t0:.1f}s")
print(f"  Removed: {n_removed:,} overlapping majority boundary samples")

# Reconstruct cleaned training set with original 3-tier labels
X_train_clean = X_unified[kept_indices]
y_train_clean = y_unified[kept_indices]

print(f"\n✓ NB-Tomek Cleaned Training Cohort: {len(y_train_clean):,} visits")
for c, lbl in enumerate(TIER_LABELS):
    orig = np.sum(y_unified == c)
    clean = np.sum(y_train_clean == c)
    print(f"  * {lbl}: {clean:,} / {orig:,} ({clean/max(orig,1)*100:.1f}% retained)")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Train Random Forest on NB-Tomek Cleaned Training Set
# ---------------------------------------------------------------------------
print("Training Random Forest on NB-Tomek Cleaned Dataset (class_weight='balanced')...")

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_clean, y_train_clean)

# Validation check
pred_val = rf_model.predict(X_val)
val_bacc = balanced_accuracy_score(y_val, pred_val)
print(f"\n✓ Random Forest Trained! Validation Balanced Accuracy: {val_bacc*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Holdout Test Set Evaluation — Recall, Specificity, Balanced Acc, ROC AUC
# ---------------------------------------------------------------------------
pred_test = rf_model.predict(X_test)
p_test    = rf_model.predict_proba(X_test)

# Overall Metrics
acc         = accuracy_score(y_test, pred_test)
bal_acc     = balanced_accuracy_score(y_test, pred_test)
macro_f1    = f1_score(y_test, pred_test, average='macro', zero_division=0)
weighted_f1 = f1_score(y_test, pred_test, average='weighted', zero_division=0)

# Per-Class Recall (Sensitivity)
recall_per = recall_score(y_test, pred_test, average=None, zero_division=0)
prec_per   = precision_score(y_test, pred_test, average=None, zero_division=0)
f1_per     = f1_score(y_test, pred_test, average=None, zero_division=0)

# Per-Class Specificity (True Negative Rate)
cm = confusion_matrix(y_test, pred_test, labels=[0, 1, 2])
specificity_per = []
for c in range(3):
    # Specificity for class c = TN / (TN + FP)
    tp = cm[c, c]
    fn = cm[c, :].sum() - tp
    fp = cm[:, c].sum() - tp
    tn = cm.sum() - tp - fn - fp
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    specificity_per.append(spec)

# ROC AUC (One-vs-Rest, macro)
y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
roc_auc_ovr = roc_auc_score(y_test_bin, p_test, average='macro', multi_class='ovr')
roc_auc_per = []
for c in range(3):
    auc_c = roc_auc_score(y_test_bin[:, c], p_test[:, c])
    roc_auc_per.append(auc_c)

# Build Report Table
report_rows = []
for c, lbl in enumerate(TIER_LABELS):
    report_rows.append({
        'Triage_Tier': lbl,
        'True_Visits': int(np.sum(y_test == c)),
        'Predicted_Visits': int(np.sum(pred_test == c)),
        'Recall (Sensitivity)': f"{recall_per[c]*100:.2f}%",
        'Specificity': f"{specificity_per[c]*100:.2f}%",
        'Precision': f"{prec_per[c]*100:.2f}%",
        'F1_Score': round(f1_per[c], 4),
        'ROC_AUC (OvR)': round(roc_auc_per[c], 4)
    })

report_df = pd.DataFrame(report_rows)

print("=" * 100)
print("  HOLDOUT TEST EVALUATION: NB-TOMEK UNDERSAMPLING + BALANCED RANDOM FOREST (3-TIER)")
print("=" * 100)
print(f"Total Test Cohort: {len(y_test):,} visits")
print(f"Overall Accuracy         : {acc*100:.2f}%")
print(f"Macro Balanced Accuracy  : {bal_acc*100:.2f}%")
print(f"Macro F1-Score           : {macro_f1:.4f}")
print(f"Weighted F1-Score        : {weighted_f1:.4f}")
print(f"Macro ROC-AUC (OvR)      : {roc_auc_ovr:.4f}\n")
print(report_df.to_string(index=False))
print("\n" + "=" * 100)

print("\nDetailed Classification Report:")
print(classification_report(y_test, pred_test, target_names=TIER_LABELS, digits=4))

# Save report
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'nb_tomek_rf_3class_report.csv')
report_df.to_csv(report_file, index=False)
print(f"Report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: 3-Class Confusion Matrix Heatmap
# ---------------------------------------------------------------------------
plots_dir = f'{ROOT}/plots'
os.makedirs(plots_dir, exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'distant_analysis'), exist_ok=True)
os.makedirs(os.path.join(plots_dir, 'image'), exist_ok=True)

cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8.5, 7))
annot = np.empty_like(cm, dtype=object)
for i in range(3):
    for j in range(3):
        annot[i, j] = f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.2f}%)"

sns.heatmap(
    cm_norm, annot=annot, fmt='', cmap='Blues', cbar=True, ax=ax,
    vmin=0, vmax=1, xticklabels=TIER_LABELS, yticklabels=TIER_LABELS
)

ax.set_title(
    f'NB-Tomek Undersampling + Balanced Random Forest: 3-Class Confusion Matrix\n'
    f'Balanced Accuracy: {bal_acc*100:.2f}% | Macro ROC-AUC: {roc_auc_ovr:.4f}',
    fontsize=11.5, fontweight='bold', pad=12
)
ax.set_xlabel('Predicted Disaster Triage Tier', fontsize=11, fontweight='bold')
ax.set_ylabel('True Disaster Triage Tier', fontsize=11, fontweight='bold')

plt.tight_layout()
cm_path = os.path.join(plots_dir, 'distant_analysis', 'nb_tomek_rf_3class_confusion_matrix.png')
plt.savefig(cm_path, dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(plots_dir, 'image', 'nb_tomek_rf_3class_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"Confusion Matrix saved to: {cm_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Feature Importance Bar Chart
# ---------------------------------------------------------------------------
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'Feature': FEATURES,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
sns.barplot(data=feat_imp_df, x='Importance', y='Feature', color='#1f77b4', ax=ax)
ax.set_title('NB-Tomek RF: Feature Importances (8 Arrival Triage Features)', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Normalized Gini Importance', fontsize=11, fontweight='bold')
ax.set_ylabel('Feature', fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
imp_path = os.path.join(plots_dir, 'distant_analysis', 'nb_tomek_rf_feature_importance.png')
plt.savefig(imp_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"Feature importance plot saved to: {imp_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 9: Export Production Bundle & Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'rf_model': rf_model,
    'features': FEATURES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'nb_tomek_rf_3class_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='NB_Tomek_Undersampling_Plus_Balanced_Random_Forest_3Class',
    nb_method='NB-Tomek (Adaptive k = sqrt(IR) + sqrt(N))',
    nb_reference='NB-undersampling/searchKnn_UnderS_Tomek.R',
    classifier='RandomForestClassifier',
    n_estimators=int(rf_model.n_estimators),
    max_depth=int(rf_model.max_depth),
    class_weight='balanced',
    features=FEATURES,
    tier_labels=TIER_LABELS,
    total_samples=len(esi_all),
    overall_accuracy=round(acc, 4),
    macro_balanced_accuracy=round(bal_acc, 4),
    macro_roc_auc_ovr=round(roc_auc_ovr, 4),
    macro_f1=round(macro_f1, 4),
    per_class_metrics=report_rows
)

manifest_file = os.path.join(deploy_dir, 'nb_tomek_rf_3class_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Bundle  : {bundle_file}")
print(f"✓ Manifest: {manifest_file}")